# Shadow Slave full-novel index rebuild (Colab) - int8 + fp32

Rebuilds the bge-large retrieval index for **all 3,160 chapters** (both
`novel_chunks` and `notebook_statements`) and zips them for Weaver.
Plan 018.5 slice 6. Run the cells **in order**.

**Before you start:**
1. Runtime -> Change runtime type -> **T4 GPU** -> Save.
2. Upload **`shadow-slave.zip`** (made locally with `zip -r shadow-slave.zip novels/shadow-slave`)
   and **`knowledge.zip`** (made locally with `zip -r knowledge.zip .weaver/knowledge/shadow-slave`)
   via the files panel.

**No notebook updates needed, ever.** The runner is downloaded fresh from GitHub
in cell 3, so fixes reach you by just re-running cell 3. Never re-upload this notebook.

**After cell 1 you must see** `IMPORTS OK:` with `CUDAExecutionProvider` and a GPU name on the torch line.

What this builds:
- **index-fp32.zip** - the baseline (bge-large fp32, the current model)
- **index-int8.zip** - the v1 target (bge-large self-quantized to int8 in colab, ~430MB)

Both use the winning 40-line story-aware chunks.

In [ ]:
# 1. Install dependencies with ONLY the GPU onnxruntime, CUDA 12 build.#    Colab's T4 is CUDA 12; the default onnxruntime-gpu (1.28+) is a CUDA 13#    build and fails to load. The CUDA 12 build lives on Microsoft's feed.!pip uninstall -y onnxruntime onnxruntime-gpu 2>/dev/null | tail -1!pip install -q fastembed qdrant-client onnx tokenizers!pip uninstall -y onnxruntime 2>/dev/null | tail -1!pip install -q onnxruntime-gpu --extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/!python -c "import fastembed, qdrant_client, onnxruntime as ort; print('IMPORTS OK:', ort.__version__, ort.get_available_providers())"!python -c "import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - Runtime > Change runtime type > T4 GPU')"!ls novels/shadow-slave | wc -l   # 33 = novel present; 0 = re-upload the zip

In [ ]:
# 2. Unzip the novel and notebook, sanity-check the layout!unzip -q -o shadow-slave.zip!unzip -q -o knowledge.zip!ls novels/shadow-slave | tail -2!ls knowledge/shadow-slave/reading | tail -2    # expect 3160.json!ls knowledge/shadow-slave/reading | wc -l      # expect 3160

In [ ]:
# 3. Fetch the LATEST runner from GitHub (no re-uploads, ever)!curl -sfL https://raw.githubusercontent.com/haxsysgit/Weaver/main/scripts/colab_build_runner.py -o colab_build_runner.py && wc -l colab_build_runner.py

## 4. Build 1 of 2: fp32 index (the baseline)

Runs the real weaver runner (real chunker, real collection creators) on CUDA.
Expect **~15-30 minutes** on a working T4, ~6900 chunks + ~2450 statements.
If CUDA fails the runner falls back to CPU automatically (slower, correct).

In [ ]:
# 4. fp32 build (CUDA on T4)!python colab_build_runner.py \    --novel-dir novels/shadow-slave \    --notebook-dir knowledge/shadow-slave \    --out /content/index-fp32 \    --ceiling 3160!du -sh /content/index-fp32

## 5. Self-quantize fp32 -> int8

`quantize_dynamic` on our trusted fp32 onnx: MatMul weights only (the
512-token position-table fix). ~30s, 1.3GB -> ~430MB. The runner needs
`tokenizer.json` beside the onnx, so we copy it from the fastembed cache.

In [ ]:
# 5. Quantize to int8 (MatMul weights only)import glob, os, shutilfrom onnxruntime.quantization import quantize_dynamic, QuantTypecache = glob.glob('/root/.cache/fastembed/models--qdrant--bge-large-en-v1.5-onnx/snapshots/*')fp32 = [p for p in cache if p.endswith('/model.onnx')][0]tok  = [p for p in cache if p.endswith('/tokenizer.json')][0]os.makedirs('/content/int8-model', exist_ok=True)quantize_dynamic(    fp32,    '/content/int8-model/model_quantized.onnx',    weight_type=QuantType.QInt8,    op_types_to_quantize=['MatMul'],)shutil.copy(tok, '/content/int8-model/tokenizer.json')print('int8 model:', round(os.path.getsize('/content/int8-model/model_quantized.onnx')/1e6), 'MB')

## 6. Build 2 of 2: int8 index (the v1 target)

Same runner, `--dense` points at the local int8 onnx (loaded directly via
onnxruntime, 512-token truncation like fastembed). Expect **~20-40 minutes**.

In [ ]:
# 6. int8 build!python colab_build_runner.py \    --novel-dir novels/shadow-slave \    --notebook-dir knowledge/shadow-slave \    --out /content/index-int8 \    --ceiling 3160 \    --dense /content/int8-model/model_quantized.onnx!du -sh /content/index-int8

## 7. Download both indexes to your machine

In [ ]:
# 7. Zip and download!cd /content && zip -q -r index-fp32.zip index-fp32!cd /content && zip -q -r index-int8.zip index-int8!ls -lh /content/index-fp32.zip /content/index-int8.zipfrom google.colab import filesfiles.download('/content/index-fp32.zip')files.download('/content/index-int8.zip')

## Back on your machine

```bash
cd /home/hax/weaver
cp -r .weaver/retrieval/index .weaver/retrieval/index-backup-ch1000   # backup first

rm -rf .weaver/retrieval/index
unzip -q ~/Downloads/index-fp32.zip -d .weaver/retrieval/

uv run python - <<'PY'
from qdrant_client import QdrantClient
c = QdrantClient(path=".weaver/retrieval/index")
for col in c.get_collections().collections:
    info = c.get_collection(col.name)
    print(col.name, "points:", info.points_count,
          "dense dim:", info.config.params.vectors.get("dense", {}).size if info.config.params.vectors else "sparse-only")
PY
```

Expect ~6900 novel chunks + ~2450 notebook statements, dense dim 1024.
Swap in `index-int8.zip` the same way for the quantization comparison.
The runtime loads the index lazily on first search.